# Stage B sơ bộ trên Kaggle

Chạy 1 panel × 3 kịch bản × W={0,4,16} × 4 phương pháp × reset={none,all}: **72 cặp, 144 lượt Q**. Giữ 1.024 ảnh lịch sử, 512 washout, 1.024 ảnh Q; severity=5; seed panel=101.

**Trước khi Run All:** bật GPU và Internet, đính kèm Dataset mã nguồn (`streaming-tta-kaggle-source.zip`) và dữ liệu ImageNet-C đã giải nén. Sửa `DATA_ROOT` và `DATA_SOURCE_URL` ở cell cấu hình. Notebook không tự tải các archive ImageNet-C lớn. GPU quota xem trong tài khoản Kaggle.

Đây là thí nghiệm thăm dò trên một panel, chưa có suy luận quần thể hoặc ablation đủ từng thành phần trạng thái. Hướng dẫn chi tiết: `research/kaggle_preliminary.md` trong bundle.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # before importing Torch
from pathlib import Path
import shutil, subprocess, sys, json, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "streaming-tta-state-memory"
# Kaggle may expose an uploaded ZIP as extracted directories.
candidates = [
    p.parent.parent for p in INPUT.rglob("kaggle_preliminary.py")
    if p.parent.name == "scripts" and (p.parent.parent / "src/historytta").is_dir()
]
if not REPO.exists():
    if len(candidates) == 1:
        shutil.copytree(candidates[0], REPO)
    elif not candidates:
        bundles = list(INPUT.rglob("streaming-tta-kaggle-source.zip"))
        if len(bundles) != 1:
            raise RuntimeError("Attach exactly one source Dataset built by scripts/create_kaggle_bundle.py.")
        with zipfile.ZipFile(bundles[0]) as archive:
            if sum(i.file_size for i in archive.infolist()) > 100_000_000:
                raise RuntimeError("Unexpectedly large source bundle.")
            for info in archive.infolist():
                path = WORK / info.filename
                if not path.resolve().is_relative_to(REPO.resolve()):
                    raise RuntimeError("Unexpected source ZIP path: " + info.filename)
                if (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise RuntimeError("Source ZIP symlinks are not supported.")
            archive.extractall(WORK)
    else:
        raise RuntimeError("Multiple source Datasets found; attach only one.")
if not (REPO / "scripts/kaggle_preliminary.py").is_file():
    raise RuntimeError("Source bundle layout is invalid.")
os.chdir(REPO)
print("Repository:", REPO)
print("Input Dataset directories:", [p.name for p in INPUT.iterdir()])


## 1. Cấu hình dữ liệu

`DATA_ROOT` phải là thư mục chứa trực tiếp `gaussian_noise/5`, `brightness/5`, `defocus_blur/5`. Mỗi variant cần 1.000 thư mục synset và cùng tập ID ảnh gốc. Chấp nhận đầy đủ ImageNet-C hoặc phần pilot đã chuẩn bị đúng hash split; không dùng ảnh synthetic/CIFAR-C hay subset chọn theo nhãn.

`DATA_SOURCE_URL` là URL **thực tế** của bộ dữ liệu đính kèm. Thiếu provenance extraction gốc thì URL là nguồn do bạn khai báo; notebook chỉ xác minh bytes của ảnh được chọn, không khẳng định đã kiểm MD5 archive chính thức.

In [ ]:
# EDIT THESE TWO VALUES.
DATA_ROOT = Path("/kaggle/input/your-imagenetc-dataset/imagenet-c")
DATA_SOURCE_URL = ""  # e.g. the actual https://www.kaggle.com/datasets/... page

CONFIG = REPO / "configs/kaggle_preliminary.yaml"
CLASS_INDEX = REPO / "datasets/metadata/imagenet_class_index.json"
DOWNLOAD_CLASS_INDEX = True  # False only if the pinned mapping is attached below.
# Offline mapping example:
# CLASS_INDEX = Path("/kaggle/input/your-metadata/imagenet_class_index.json")

MAX_HOURS = 10.0  # Runner subprocess wall-clock cap; not a measured runtime.
EXPORT_DIR = WORK / "kaggle_exports"

if not DATA_ROOT.is_dir():
    raise FileNotFoundError("Edit DATA_ROOT to the actual extracted ImageNet-C folder.")
for domain in ["gaussian_noise", "brightness", "defocus_blur"]:
    variant = DATA_ROOT / domain / "5"
    if not variant.is_dir():
        raise FileNotFoundError(variant)
    print(domain, "synset directories:", sum(p.is_dir() for p in variant.iterdir()))


## 2. Kiểm tra môi trường

Giữ bộ Torch/torchvision có sẵn của Kaggle, không cài đè CUDA wheels. Chỉ cài các gói nhỏ còn thiếu. Nếu import Torch/torchvision lỗi, sửa môi trường trước khi chạy. Internet dùng để lấy mapping 35 KB, trọng số ResNet50 và gói còn thiếu; không tải ImageNet-C.

In [ ]:
import importlib.util
for module in ["torch", "torchvision"]:
    if importlib.util.find_spec(module) is None:
        raise RuntimeError("Kaggle GPU image must provide " + module)
packages = {"yaml": "PyYAML", "numpy": "numpy", "pandas": "pandas",
            "scipy": "scipy", "matplotlib": "matplotlib", "PIL": "Pillow", "pytest": "pytest"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
import torch, torchvision
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle Notebook Settings.")
print("Torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda, "GPU:", torch.cuda.get_device_name(0))
torch.set_num_threads(2)
probe = torch.ones((8, 8), device="cuda")
print("CUDA smoke:", (probe @ probe).mean().item())
del probe
torch.cuda.empty_cache()


## 3. Xem kế hoạch và chạy kiểm tra nhỏ

Các test dưới đây không tải trọng số và không đọc dữ liệu ImageNet-C. Kiểm tra runner/reset/audit trên fixtures trước khi chạy thật.

In [ ]:
def helper(*arguments):
    subprocess.run([sys.executable, "-u", "scripts/kaggle_preliminary.py",
                    *map(str, arguments), "--config", str(CONFIG)], cwd=REPO, check=True)

helper("plan")
subprocess.run([sys.executable, "-m", "pytest", "-q",
                "tests/test_cloud_plan.py", "tests/test_adapters.py",
                "tests/test_protocol.py", "tests/test_evaluate.py",
                "tests/test_kaggle_preliminary.py"], cwd=REPO, check=True)


## 4. Tạo manifest theo đường dẫn Kaggle

Sampler chỉ dùng tên ảnh và hash. Không chọn lớp cân bằng, không mượn ảnh thuộc reserve. Cần 2.560 ID pilot khác nhau (7.680 file ảnh qua ba corruption). Tạo manifest idempotent khi mọi dữ liệu và đường dẫn giữ nguyên.

In [ ]:
arguments = ["prepare", "--data-root", DATA_ROOT, "--source-url", DATA_SOURCE_URL,
             "--class-index", CLASS_INDEX]
if DOWNLOAD_CLASS_INDEX:
    arguments.append("--download-class-index")
helper(*arguments)


## 5. Chạy thật, audit, tạo bảng/hình và xuất ZIP

Cell này tải trọng số ResNet50 V1 nếu chưa cache, chạy 144 lượt Q và deterministic controls. Không tự dùng cả hai GPU nếu Kaggle cấp hai GPU. Sau khi đủ lượt chạy và audit/controls đạt, tạo bảng, hình và CSV sơ bộ; bảng factorial được bỏ qua vì chỉ có none/all.

`finally` đóng gói cả khi runner lỗi hoặc hết giới hạn subprocess. Nếu Kaggle dừng toàn bộ session, cell cuối có thể chưa thực thi; không xem kết quả thiếu lượt là kết quả hoàn chỉnh. Để chạy lại, sửa **cả experiment_id và output** trong YAML; giữ manifest nếu dữ liệu không đổi.

In [ ]:
try:
    helper("run", "--max-hours", MAX_HOURS)
finally:
    helper("export", "--export-dir", EXPORT_DIR)

from IPython.display import FileLink, display
cfg = __import__("yaml").safe_load(CONFIG.read_text())
archive = EXPORT_DIR / (cfg["experiment_id"] + "_artifacts.zip")
display(FileLink(str(archive.relative_to(WORK))))


## 6. Đọc kết quả

- `run_status.json`: phải `complete`.
- `evaluation_audit.json`: phải `passed: true`.
- `controls.json`: mọi `passed` phải true.
- `preliminary_history.csv`: AB/BA disagreement theo kịch bản, W, phương pháp, reset. `disagreement_pp` là điểm phần trăm.
- `preliminary_performance.csv`: accuracy/loss trung bình hai chiều, giữ từng kịch bản riêng.
- `tables/` và `figures/`: bảng và hình từ dữ liệu thật.

Không kết luận một phương pháp tốt hơn ở quần thể từ một panel. Disagreement bằng 0 vẫn có thể có loss/logits khác nhau.

In [ ]:
import pandas as pd
cfg = __import__("yaml").safe_load(CONFIG.read_text())
RESULTS = REPO / cfg["output"]
status = json.loads((RESULTS / "run_status.json").read_text())
audit = json.loads((RESULTS / "evaluation_audit.json").read_text())
controls = json.loads((RESULTS / "controls.json").read_text())
assert status["status"] == "complete" and audit["passed"]
assert controls and all(c["passed"] for c in controls)
print("Run status:", status)
print("Audit:", audit["counts"])
history = pd.read_csv(RESULTS / "preliminary_history.csv")
display(history[history.washout_batches == 16])
performance = pd.read_csv(RESULTS / "preliminary_performance.csv")
display(performance[(performance.washout_batches == 16) & (performance.intervention == "none")])
from IPython.display import FileLink, display
archive = EXPORT_DIR / (cfg["experiment_id"] + "_artifacts.zip")
display(FileLink(str(archive.relative_to(WORK))))
